# Conversion Rate Challenge
## Predict newsletter subscriptions using machine learning

**Objective:** Build a model to predict whether a website visitor will subscribe to the newsletter. Analyze model parameters to identify levers for improving conversion rate.

**Metric:** F1-score

**Approach:**
1. Exploratory Data Analysis
2. Feature Engineering
3. Model training (Logistic Regression baseline, Random Forest, GradientBoosting with GridSearchCV)
4. Threshold optimization (balancing false positives vs false negatives)
5. Business recommendations

In [ ]:
import sys
import warnings

sys.path.insert(0, ".")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm import tqdm

from src.config import (
    METADATA_FILE,
    MODEL_FILE,
    MODELS_DIR,
    PREPROCESSOR_FILE,
    RANDOM_STATE,
    TARGET,
    TEST_FILE,
    TRAIN_FILE,
)
from src.features import (
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    add_features,
)

# Export directory for plot images
PLOT_DIR = Path("reports/figures")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# WHY: ensures reproducibility for any operation using numpy's global random
# state (e.g., train_test_split uses it internally)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# 1. Data Loading

In [2]:
train_data = pd.read_csv(TRAIN_FILE)
test_data = pd.read_csv(TEST_FILE)

print(f"Train: {train_data.shape[0]} rows, {train_data.shape[1]} columns")
print(f"Test:  {test_data.shape[0]} rows, {test_data.shape[1]} columns")
print(f"\nColumns (train): {list(train_data.columns)}")
print(f"Columns (test):  {list(test_data.columns)}")
print(f"\nMissing values: {train_data.isnull().sum().sum()}")

Train: 284580 rows, 6 columns
Test:  31620 rows, 5 columns

Columns (train): ['country', 'age', 'new_user', 'source', 'total_pages_visited', 'converted']
Columns (test):  ['country', 'age', 'new_user', 'source', 'total_pages_visited']

Missing values: 0


In [3]:
display(train_data.head(10))
display(train_data.describe())
display(train_data.dtypes)

,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0
5,US,29,0,Seo,7,0
6,US,30,1,Direct,4,0
7,UK,38,1,Ads,2,0
8,UK,26,1,Seo,4,0
9,UK,31,0,Seo,5,0


,age,new_user,total_pages_visited,converted
count,284580.000000,284580.000000,284580.000000,284580.000000
mean,30.564203,0.685452,4.873252,0.032258
std,8.266789,0.464336,3.341995,0.176685
min,17.000000,0.000000,1.000000,0.000000
25%,24.000000,0.000000,2.000000,0.000000
50%,30.000000,1.000000,4.000000,0.000000
75%,36.000000,1.000000,7.000000,0.000000
max,123.000000,1.000000,29.000000,1.000000


country                object
age                     int64
new_user                int64
source                 object
total_pages_visited     int64
converted               int64
dtype: object

### Observations from descriptive statistics

**Dataset size:** 284,580 rows, 6 columns. No missing values detected.

**Target variable (`converted`):** mean = 0.032 confirms the 3.2% conversion rate. The 75th percentile is still 0, reflecting the severe class imbalance (~30:1 ratio). Accuracy would be misleading here; F1-score is the appropriate metric.

**Age:** ranges from 17 to 123. The maximum of 123 (and likely 111+) are data entry errors requiring cleanup -- a threshold around 80 removes implausible values. The core distribution is roughly symmetric (mean 30.6 ~ median 30.0), with IQR [24, 36].

**Total pages visited:** right-skewed (mean 4.87 > median 4.0), with a long tail up to 29 pages. IQR [2, 7] shows that 50% of users visit very few pages. This skewness suggests the feature may benefit from binning or log-transform for linear models.

**New user:** binary variable. Mean = 0.69 indicates ~69% new visitors vs ~31% returning. No transformation needed.

**Note:** only numerical columns appear in `describe()`. The two categorical features (`country`, `source`) are analyzed in section 2.

# 2. Exploratory Data Analysis

## 2.1 Target Distribution

The dataset is highly imbalanced: only ~3.2% of visitors convert. This will require careful handling during modeling (threshold tuning).

In [4]:
target_counts = train_data[TARGET].value_counts()
target_pct = train_data[TARGET].value_counts(normalize=True) * 100
imbalance_ratio = target_counts[0] / target_counts[1]

fig = go.Figure(data=[
    go.Bar(
        x=["Not Converted (0)", "Converted (1)"],
        y=target_counts.values,
        marker_color=["#3498db", "#e74c3c"],
        text=[
            f"{v} ({p:.1f}%)"
            for v, p in zip(target_counts.values, target_pct.values)
        ],
        textposition="auto",
    ),
])

fig.update_layout(
    title_text=f"Target Distribution (Imbalance ratio: {imbalance_ratio:.0f}:1)",
    xaxis_title="Class",
    yaxis_title="Count",
    height=450,
    width=600,
)
fig.write_image(PLOT_DIR / "01_target_distribution.png", width=600, height=450, scale=2)
fig.show()

### Expected output

![01_target_distribution](reports/figures/01_target_distribution.png)

- The dataset is heavily imbalanced with a 30:1 ratio: 275,400 non-converted (96.8%) vs 9,180 converted (3.2%).
- A naive "always predict 0" baseline would yield 96.8% accuracy but 0% F1 -- confirming that F1 is the right metric for this problem.
- With only ~9k positive samples, stratified splitting will be essential to preserve class proportions in train/validation sets.

## 2.2 Feature Distributions

In [5]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Age Distribution", "Total Pages Visited"),
)

fig.add_trace(
    go.Histogram(
        x=train_data["age"], nbinsx=30,
        marker_color="#3498db", name="Age",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(
        x=train_data["total_pages_visited"], nbinsx=20,
        marker_color="#e74c3c", name="Pages",
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text="Age", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_xaxes(title_text="Total Pages Visited", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.update_layout(
    height=400, showlegend=False,
    title_text="Numerical Feature Distributions",
)
fig.write_image(PLOT_DIR / "03_numerical_distributions.png", width=1200, height=400, scale=2)
fig.show()

### Expected output

![03_numerical_distributions](reports/figures/03_numerical_distributions.png)

- Age distribution is right-skewed with the bulk of users between 20-45 and a long tail extending beyond 100, suggesting data entry errors at extreme ages.
- Total pages visited is heavily right-skewed: most users visit 1-5 pages with a steep drop-off. Very few users visit more than 15 pages.
- The page visit distribution hints at a clear engagement threshold -- users who go beyond ~10 pages may represent a qualitatively different, more engaged population.

In [6]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Country", "Source", "New User"),
)

# Readable labels for new_user
new_user_labels = train_data["new_user"].map(
    {0: "Returning", 1: "New User"}
)

feature_names = ["Country", "Source", "New User"]

for i, (col, data) in enumerate([
    ("country", train_data["country"]),
    ("source", train_data["source"]),
    ("new_user", new_user_labels),
]):
    counts = data.value_counts()
    total = counts.sum()
    percentages = counts / total * 100
    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            marker_color="#3498db",
            text=[f"{p:.1f}%" for p in percentages],
            textposition="auto",
        ),
        row=1, col=i + 1,
    )
    fig.update_xaxes(title_text=feature_names[i], row=1, col=i + 1)
    fig.update_yaxes(title_text="Count", row=1, col=i + 1)

fig.update_layout(
    height=400, showlegend=False,
    title_text="Categorical Feature Distributions",
)
fig.write_image(PLOT_DIR / "02_categorical_distributions.png", width=1200, height=400, scale=2)
fig.show()

### Expected output

![02_categorical_distributions](reports/figures/02_categorical_distributions.png)

- US dominates the dataset (~56%), followed by China (~24%), UK (~15%), and Germany (~4%). The strong US skew means country-level patterns for Germany will be estimated on fewer samples.
- Traffic source is relatively balanced: Seo leads (~49%), then Ads (~28%), then Direct (~23%).
- New users represent ~69% of the audience vs ~31% returning users, indicating the site acquires a lot of first-time visitors whose conversion behavior may differ significantly.

## 2.3 Conversion Rate by Feature

Understanding which features drive conversion is key to both modeling and business recommendations.

In [7]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "By Country", "By Source", "By User Type",
    ),
)

avg_rate = train_data[TARGET].mean()

# Readable labels for new_user
user_type = train_data["new_user"].map(
    {0: "Returning", 1: "New User"}
)

# WHY: track whether the first trace of each color has been added
# to avoid duplicate legend entries across subplots
first_above = True
first_below = True

for i, (col, data) in enumerate([
    ("country", train_data["country"]),
    ("source", train_data["source"]),
    ("new_user", user_type),
]):
    conv_rate = (
        pd.DataFrame({
            "group": data,
            TARGET: train_data[TARGET],
        })
        .groupby("group")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )

    above_mask = conv_rate.values > avg_rate
    below_mask = ~above_mask

    # Above average bars
    if above_mask.any():
        fig.add_trace(
            go.Bar(
                x=conv_rate.index[above_mask].astype(str),
                y=conv_rate.values[above_mask],
                marker_color="#2ecc71",
                text=[f"{v:.1%}" for v in conv_rate.values[above_mask]],
                textposition="auto",
                name="Above average",
                showlegend=first_above,
                legendgroup="above",
            ),
            row=1, col=i + 1,
        )
        first_above = False

    # Below average bars
    if below_mask.any():
        fig.add_trace(
            go.Bar(
                x=conv_rate.index[below_mask].astype(str),
                y=conv_rate.values[below_mask],
                marker_color="#3498db",
                text=[f"{v:.1%}" for v in conv_rate.values[below_mask]],
                textposition="auto",
                name="Below average",
                showlegend=first_below,
                legendgroup="below",
            ),
            row=1, col=i + 1,
        )
        first_below = False

    fig.add_hline(
        y=avg_rate, line_dash="dash",
        line_color="red",
        annotation_text=f"avg: {avg_rate:.1%}",
        row=1, col=i + 1,
    )
    fig.update_yaxes(title_text="Conversion Rate", row=1, col=i + 1)

fig.update_layout(
    height=450,
    title_text="Conversion Rate by Feature",
)
fig.update_yaxes(tickformat=".1%")
fig.write_image(PLOT_DIR / "04_conversion_rate_by_feature.png", width=1200, height=450, scale=2)
fig.show()

### Expected output

![04_conversion_rate_by_feature](reports/figures/04_conversion_rate_by_feature.png)

- Germany (6.2%) and UK (5.2%) have conversion rates well above the 3.2% average, while China is an extreme outlier at 0.1%. Country (specifically China vs. rest) is a strong discriminative feature.
- Source differences are modest (Ads 3.5%, Seo 3.3%, Direct 2.8%) and unlikely to be a top predictor.
- Returning users convert at 7.2% vs 1.4% for new users -- a 5x difference and the second strongest signal after page visits. Retention strategies should be prioritized.

In [8]:
pages_conv = (
    train_data
    .groupby("total_pages_visited")[TARGET]
    .agg(["mean", "count"])
    .reset_index()
)
pages_conv.columns = [
    "total_pages_visited", "conversion_rate", "count",
]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=pages_conv["total_pages_visited"],
        y=pages_conv["count"],
        name="Visitor Count",
        marker_color="#3498db",
        opacity=0.5,
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=pages_conv["total_pages_visited"],
        y=pages_conv["conversion_rate"],
        name="Conversion Rate",
        line=dict(color="#e74c3c", width=3),
        mode="lines+markers",
    ),
    secondary_y=True,
)

fig.update_layout(
    title_text=(
        "Conversion Rate vs Total Pages Visited (Key Predictor)"
    ),
    height=450,
)
fig.update_yaxes(title_text="Visitor Count", secondary_y=False)
fig.update_yaxes(
    title_text="Conversion Rate",
    tickformat=".0%",
    secondary_y=True,
)
fig.write_image(PLOT_DIR / "05_conversion_vs_pages.png", width=800, height=450, scale=2)
fig.show()

### Expected output

![05_conversion_vs_pages](reports/figures/05_conversion_vs_pages.png)

- Conversion rate follows a sigmoid curve, jumping from near 0% at 1-5 pages to nearly 100% at 15+ pages. Total pages visited is clearly the dominant predictor.
- The inflection point is around 8-12 pages -- users who cross this threshold are dramatically more likely to convert. This is a concrete business lever: design the site to encourage deeper browsing.
- The vast majority of traffic sits in the 1-5 pages range (low conversion zone), meaning even a small shift in engagement depth could yield significant conversion gains.

In [9]:
age_conv = (
    train_data.groupby("age")[TARGET].mean().reset_index()
)
age_conv.columns = ["age", "conversion_rate"]

fig = px.scatter(
    age_conv, x="age", y="conversion_rate",
    title="Conversion Rate by Age",
    labels={
        "conversion_rate": "Conversion Rate",
        "age": "Age",
    },
)
fig.update_traces(marker=dict(size=8, color="#3498db"))
fig.update_yaxes(tickformat=".1%")
fig.add_hline(
    y=avg_rate, line_dash="dash", line_color="red",
    annotation_text=f"avg: {avg_rate:.1%}",
)
fig.write_image(PLOT_DIR / "06_conversion_by_age.png", width=800, height=450, scale=2)
fig.show()

### Expected output

![06_conversion_by_age](reports/figures/06_conversion_by_age.png)

- Young users (age < 25) show the highest conversion rates (~5-10%), with a monotonic decline as age increases. Users aged 40+ convert at or below the 3.2% average.
- Two data points at extreme ages (~111 and ~123) show 100% conversion rate -- these are data quality anomalies and must be removed before modeling.
- The smooth, monotonically decreasing relationship suggests a simple transform of age could capture this effect well.

## 2.4 Correlation Analysis

In [10]:
numeric_cols = [
    "age", "new_user", "total_pages_visited", TARGET,
]
corr_matrix = train_data[numeric_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    text=corr_matrix.round(3).values,
    texttemplate="%{text}",
    colorscale="RdBu_r",
    zmid=0,
))
fig.update_layout(
    title="Correlation Matrix",
    width=700,
    height=600,
)
fig.write_image(PLOT_DIR / "07_correlation_matrix.png", width=700, height=600, scale=2)
fig.show()

print("\nCorrelation with target:")
print(corr_matrix[TARGET].sort_values(ascending=False).to_string())


Correlation with target:
converted              1.000000
total_pages_visited    0.529192
age                   -0.088265
new_user              -0.152115


### Expected output

![07_correlation_matrix](reports/figures/07_correlation_matrix.png)

- `total_pages_visited` has the strongest correlation with `converted` (0.529), confirming it as the dominant linear predictor.
- `new_user` has a moderate negative correlation (-0.152), aligning with the 7.2% vs 1.4% conversion gap observed above.
- Features are largely uncorrelated with each other (all pairwise values < 0.1), which is favorable: minimal multicollinearity means each feature brings independent information.
- Note: Pearson correlation only captures linear relationships. The sigmoid relationship between pages and conversion is actually stronger than 0.529 suggests.

## 2.5 EDA Key Findings

1. **Severe class imbalance** — 96.8% / 3.2% split (30:1 ratio)
2. **`total_pages_visited`** is the dominant predictor (corr = 0.53). Conversion rate increases sharply with pages visited.
3. **Returning users** convert ~5x more than new users (7.2% vs 1.4%)
4. **Germany** has the highest conversion rate (~6.2%), **China** the lowest (~0.1%)
5. **Traffic sources:** SEO 49.0%, Ads 28.1%, Direct 22.9%. Conversion by source: Ads 3.5%, SEO 3.3%, Direct 2.8% — differences are small.
6. **Country conversion spread:** Germany 6.2%, UK 5.2%, US 3.8%, China 0.1% — China is a clear outlier.
7. **Age** has weak negative correlation with conversion — older users slightly less likely to convert
8. No missing values in any column
9. **Outlier detected**: age max = 123 (likely data entry error — to investigate)
10. **Decision:** Rows with age > 80 are dropped as data entry errors (2 rows removed)

**Modeling implications:**
- Must handle class imbalance — using threshold optimization for best F1
- `total_pages_visited` will dominate — feature engineering on other variables may help
- Business context: a **false negative** (missing a potential subscriber) is arguably worse than a **false positive** (targeting a non-subscriber) — this should influence threshold selection

# 3. Feature Engineering

In [11]:
train_enhanced = add_features(train_data)
test_enhanced = add_features(test_data)

print(f"Original features: {train_data.shape[1]}")
print(f"Enhanced features: {train_enhanced.shape[1]}")
print(
    f"\nEngineered features: "
    f"{[c for c in train_enhanced.columns if c not in train_data.columns]}"
)

Original features: 6
Enhanced features: 13

Engineered features: ['age_group', 'pages_group', 'new_user_x_pages', 'age_x_pages', 'is_germany', 'is_china', 'is_seo']


In [12]:
numeric_cols = train_enhanced.select_dtypes(include=[np.number]).columns
correlations = (
    train_enhanced[numeric_cols].corr()[TARGET]
    .sort_values(ascending=False)
)

# WHY: two separate traces needed so Plotly generates a legend entry per color
corr_vals = correlations.values[1:]
corr_names = correlations.index[1:]
pos_mask = corr_vals > 0
neg_mask = ~pos_mask

fig = go.Figure()

fig.add_trace(go.Bar(
    x=corr_vals[pos_mask],
    y=corr_names[pos_mask],
    orientation="h",
    marker_color="#2ecc71",
    name="Positive correlation",
))

fig.add_trace(go.Bar(
    x=corr_vals[neg_mask],
    y=corr_names[neg_mask],
    orientation="h",
    marker_color="#e74c3c",
    name="Negative correlation",
))

fig.update_layout(
    title="Feature Correlation with Target",
    height=400,
    xaxis_title="Correlation",
    yaxis_title="Feature",
)
fig.write_image(PLOT_DIR / "08_feature_correlation.png", width=800, height=400, scale=2)
fig.show()

### Expected output

![08_feature_correlation](reports/figures/08_feature_correlation.png)

- `total_pages_visited` has the strongest positive correlation with conversion (~0.53), confirming page engagement as the primary linear predictor.
- The interaction feature `age_x_pages` also shows positive correlation, but it is dominated by the `total_pages_visited` component (since `age_x_pages = age * total_pages_visited`). Its apparent strength largely reflects the page-visit signal rather than an independent age effect.
- `new_user` and `is_china` are the strongest negative correlators, reinforcing the importance of user retention and the China anomaly.

### Feature Selection Analysis

Before training, we analyze which features contribute most to predicting conversion, using mutual information scores.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Prepare features for MI analysis (encode categoricals)
mi_df = train_enhanced.drop(TARGET, axis=1).copy()
mi_target = train_enhanced[TARGET].copy()

# Encode categoricals for MI computation
for col in CATEGORICAL_FEATURES:
    mi_df[col] = mi_df[col].astype("category").cat.codes

mi_scores = mutual_info_classif(
    mi_df, mi_target, random_state=RANDOM_STATE
)
mi_results = (
    pd.DataFrame(
        {"feature": mi_df.columns, "mi_score": mi_scores}
    )
    .sort_values("mi_score", ascending=False)
    .reset_index(drop=True)
)

print("Mutual Information Scores:")
print(mi_results.to_string(index=False))
print(
    "\nAll features retained — tree models handle "
    "irrelevant features natively via split selection."
)

# WHY: two separate traces so Plotly generates a legend entry per color
informative_mask = mi_results["mi_score"] > 0.01
weak_mask = ~informative_mask

fig = go.Figure()

fig.add_trace(go.Bar(
    x=mi_results.loc[informative_mask, "mi_score"],
    y=mi_results.loc[informative_mask, "feature"],
    orientation="h",
    marker_color="#2ecc71",
    name="MI > 0.01 (informative)",
))

fig.add_trace(go.Bar(
    x=mi_results.loc[weak_mask, "mi_score"],
    y=mi_results.loc[weak_mask, "feature"],
    orientation="h",
    marker_color="#95a5a6",
    name="MI <= 0.01 (weak signal)",
))

fig.update_layout(
    title="Feature Selection: Mutual Information Scores",
    xaxis_title="Mutual Information",
    yaxis_title="Feature",
    yaxis=dict(autorange="reversed"),
    height=400,
)
fig.write_image(PLOT_DIR / "09_mutual_information.png", width=800, height=400, scale=2)
fig.show()

### Expected output

![09_mutual_information](reports/figures/09_mutual_information.png)

- `pages_group` and `total_pages_visited` dominate MI scores, confirming that page engagement features carry the most predictive information regardless of relationship type.
- `new_user` ranks 3rd with a higher MI score than its linear correlation suggests, indicating a non-linear (threshold) effect on conversion.
- `age` (MI ≈ 0.006) and `is_germany` (MI ≈ 0.001) fall below the 0.01 threshold, confirming both are uninformative by linear and non-linear measures.

# 4. Data Preparation

In [14]:
X_full = train_enhanced.drop(TARGET, axis=1)
y_full = train_enhanced[TARGET]
X_test_final = test_enhanced.copy()

# Threshold at 80: ages 111 and 123 are clearly data entry errors,
# and 80 is a reasonable upper bound for newsletter readership demographics
age_mask = X_full["age"] <= 80
X_full = X_full[age_mask]
y_full = y_full[age_mask]
print(f"Dropped {(~age_mask).sum()} rows with age > 80")

test_ratio = len(test_data) / len(train_data)
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size=test_ratio,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"Train conversion rate: {y_train.mean():.4f}")
print(f"Val conversion rate:   {y_val.mean():.4f}")

Dropped 2 rows with age > 80
X_train: (252958, 12)
X_val:   (31620, 12)
Train conversion rate: 0.0323
Val conversion rate:   0.0323


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
        ("num", StandardScaler(), NUMERICAL_FEATURES),
        ("bin", "passthrough", BINARY_FEATURES),
    ]
)

# 5. Model Training & Comparison

We train three models progressively:
1. **Logistic Regression** -- interpretable baseline
2. **Random Forest** -- ensemble baseline
3. **GradientBoosting** -- gradient boosting (expected best performer)

**Class imbalance strategy:** We intentionally do NOT use `class_weight='balanced'` or heavy class weighting. The ~30:1 imbalance ratio means 'balanced' would apply ~30x weight to the minority class, which crushes precision and lowers F1 (empirically, LR drops from ~0.75 to ~0.72). Instead, we optimize the decision threshold in section 7, which improves F1 by shifting the precision-recall trade-off without distorting the model's probability estimates. A break-even analysis (precision = recall) is also provided as a business-oriented alternative.

### Why these models?

**Logistic Regression (baseline):** The natural starting point for binary classification -- it provides a fully interpretable baseline where coefficients directly reveal each feature's linear impact on conversion. Fast to train on 285k rows and sets a performance floor that more complex models must demonstrably beat.

**Random Forest:** Captures non-linear relationships and feature interactions that Logistic Regression misses (e.g., how page visits interact with traffic source). As a bagging ensemble, it is robust to noise and provides built-in feature importance. Handles mixed feature types without requiring careful scaling.

**Gradient Boosting (best model):** Builds trees sequentially, with each tree focusing on previous errors -- particularly effective at learning the rare positive class (3.2%). Typically achieves higher F1 on structured tabular data than Random Forest because it optimizes a loss function directly. Its hyperparameters offer fine-grained control to balance precision and recall.

**Model progression rationale:** Starting linear, then bagging, then boosting isolates the source of each performance gain. Logistic Regression tests linear separability; Random Forest tests whether non-linearity helps; Gradient Boosting tests whether sequential error correction pushes further. This ensures added complexity is justified by measurable improvement.

In [16]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=1000,
        )),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,  # enough trees for stable predictions without excessive training time
            max_depth=15,  # deep enough to capture interactions, shallow enough to avoid overfitting
            min_samples_split=5,  # prevents splits on very small groups, reduces noise
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "GradientBoosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(
            n_estimators=100,  # conservative starting point for baseline comparison
            max_depth=3,  # shallow trees typical for boosting (sequential correction)
            learning_rate=0.05,  # low rate for better generalization, compensated by n_estimators
            random_state=RANDOM_STATE,
        )),
    ]),
}

results = []
fitted_models = {}

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    y_val_pred = pipeline.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred)
    val_recall = recall_score(y_val, y_val_pred)
    results.append({
        "Model": name,
        "F1": val_f1,
        "Precision": val_precision,
        "Recall": val_recall,
    })
    fitted_models[name] = pipeline
    print(
        f"  F1={val_f1:.4f}  "
        f"Precision={val_precision:.4f}  "
        f"Recall={val_recall:.4f}"
    )

print("\nDone.")

Training Logistic Regression...
  F1=0.7548  Precision=0.8666  Recall=0.6686
Training Random Forest...
  F1=0.7432  Precision=0.8305  Recall=0.6725
Training GradientBoosting...
  F1=0.7593  Precision=0.8589  Recall=0.6804

Done.


## 5.1 GradientBoosting -- Hyperparameter Tuning (Manual Grid Search)

GradientBoosting was the best performer at default settings. We tune its key hyperparameters via a manual grid search (`ParameterGrid` + `cross_val_score` loop with tqdm), using F1-score as the scoring metric.

> **Note:** Hyperparameter tuning does not always improve performance.
> Overfitting on cross-validation folds can lead to a tuned model
> that performs worse on held-out validation data.

In [17]:
gbt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    )),
])

# Search space chosen around baseline values:
# - n_estimators: 100-300 range balances training time vs performance
# - max_depth: 2-4 keeps trees shallow (boosting best practice)
# - learning_rate: 0.01-0.1 explores the low-rate / high-estimator tradeoff
# - min_samples_split: 5-10 prevents overfitting on small leaf nodes
gbt_param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [2, 3, 4],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__min_samples_split": [5, 10],
}

param_list = list(ParameterGrid(gbt_param_grid))
n_combos = len(param_list)
print(f"GBT GridSearchCV: {n_combos} combinations x 3 folds = {n_combos * 3} fits")

best_score = -1
best_params = None
all_results = []

# GBT is inherently sequential; parallelizing folds adds overhead without speedup
for params in tqdm(param_list, desc="GBT GridSearchCV", unit="combo"):
    gbt_pipeline.set_params(**params)
    scores = cross_val_score(
        gbt_pipeline, X_train, y_train, cv=3, scoring="f1", n_jobs=1,
    )
    mean_score = scores.mean()
    all_results.append({"params": params, "mean_f1": mean_score, "std_f1": scores.std()})
    if mean_score > best_score:
        best_score = mean_score
        best_params = params

# Apply best parameters and refit on full training set
gbt_pipeline.set_params(**best_params)
gbt_pipeline.fit(X_train, y_train)

print(f"\nBest F1: {best_score:.4f}")
print(f"Best params: {best_params}")

y_val_pred_gbt = gbt_pipeline.predict(X_val)
gbt_f1 = f1_score(y_val, y_val_pred_gbt)
gbt_precision = precision_score(y_val, y_val_pred_gbt)
gbt_recall = recall_score(y_val, y_val_pred_gbt)

results.append({
    "Model": "GradientBoosting (tuned)",
    "F1": gbt_f1,
    "Precision": gbt_precision,
    "Recall": gbt_recall,
})
fitted_models["GradientBoosting (tuned)"] = gbt_pipeline

print(
    f"Validation F1={gbt_f1:.4f}  "
    f"Precision={gbt_precision:.4f}  "
    f"Recall={gbt_recall:.4f}"
)

GBT GridSearchCV: 54 combinations x 3 folds = 162 fits


GBT GridSearchCV: 100%|██████████| 54/54 [1:10:58<00:00, 78.85s/combo] 



Best F1: 0.7643
Best params: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 300}
Validation F1=0.7552  Precision=0.8627  Recall=0.6716


### Logistic Regression -- Hyperparameter Tuning (GridSearchCV)

Tuning regularization strength (`C`) and penalty type to potentially improve the baseline logistic regression.

In [18]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=1000,  # convergence safety margin for saga solver
            solver="saga",  # required for L1 penalty support (liblinear alternative is slower on large datasets)
        ),
    ),
])
lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__penalty": ["l1", "l2"],
}
lr_grid = GridSearchCV(
    lr_pipeline,
    param_grid=lr_param_grid,
    cv=3,
    scoring="f1",
    verbose=2,
    n_jobs=-1,
)
lr_grid.fit(X_train, y_train)

lr_best = lr_grid.best_estimator_
lr_val_pred = lr_best.predict(X_val)
lr_val_f1 = f1_score(y_val, lr_val_pred)

print(f"Best LR params: {lr_grid.best_params_}")
print(f"Best LR CV F1: {lr_grid.best_score_:.4f}")
print(f"LR Validation F1: {lr_val_f1:.4f}")

results.append({
    "Model": "Logistic Regression (tuned)",
    "F1": lr_val_f1,
    "Precision": precision_score(
        y_val, lr_val_pred
    ),
    "Recall": recall_score(y_val, lr_val_pred),
})
fitted_models["Logistic Regression (tuned)"] = lr_best

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END .........classifier__C=0.01, classifier__penalty=l1; total time=   2.1s
[CV] END .........classifier__C=0.01, classifier__penalty=l1; total time=   2.2s
[CV] END .........classifier__C=0.01, classifier__penalty=l1; total time=   2.4s
[CV] END .........classifier__C=0.01, classifier__penalty=l2; total time=   2.5s
[CV] END .........classifier__C=0.01, classifier__penalty=l2; total time=   2.4s
[CV] END .........classifier__C=0.01, classifier__penalty=l2; total time=   2.5s
[CV] END ..........classifier__C=0.1, classifier__penalty=l1; total time=   3.7s
[CV] END ..........classifier__C=0.1, classifier__penalty=l1; total time=   4.2s
[CV] END ..........classifier__C=0.1, classifier__penalty=l1; total time=   4.4s
[CV] END ..........classifier__C=1.0, classifier__penalty=l1; total time=   5.6s
[CV] END ..........classifier__C=1.0, classifier__penalty=l1; total time=   7.2s
[CV] END ..........classifier__C=1.0, classifier_

In [19]:
results_df = pd.DataFrame(results).sort_values(
    "F1", ascending=True,
)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=results_df["Model"], x=results_df["F1"],
    name="F1", orientation="h", marker_color="#2ecc71",
))
fig.add_trace(go.Bar(
    y=results_df["Model"], x=results_df["Precision"],
    name="Precision", orientation="h", marker_color="#3498db",
))
fig.add_trace(go.Bar(
    y=results_df["Model"], x=results_df["Recall"],
    name="Recall", orientation="h", marker_color="#e74c3c",
))

fig.update_layout(
    barmode="group",
    title="Model Comparison",
    xaxis_title="Score",
    yaxis_title="Model",
    height=400,
)
fig.write_image(PLOT_DIR / "10_model_comparison.png", width=1000, height=400, scale=2)
fig.show()

print(results_df.to_string(index=False))

                      Model       F1  Precision   Recall
              Random Forest 0.743229   0.830508 0.672549
Logistic Regression (tuned) 0.754425   0.865482 0.668627
        Logistic Regression 0.754842   0.866582 0.668627
   GradientBoosting (tuned) 0.755237   0.862720 0.671569
           GradientBoosting 0.759300   0.858911 0.680392


### Expected output

![10_model_comparison](reports/figures/10_model_comparison.png)

**Model comparison (validation set, threshold 0.5):**

| Model | F1 | Precision | Recall |
|---|---|---|---|
| Logistic Regression (baseline) | 0.7548 | 0.8666 | 0.6686 |
| Random Forest (baseline) | 0.7432 | 0.8305 | 0.6725 |
| GradientBoosting (baseline) | **0.7593** | 0.8589 | 0.6804 |
| GradientBoosting (tuned) | 0.7552 | 0.8627 | 0.6716 |
| Logistic Regression (tuned) | 0.7544 | 0.8655 | 0.6686 |

- All five models achieve F1 scores in a very narrow band (**0.7432--0.7593**), suggesting the dataset's signal ceiling is largely reached regardless of model choice.
- **GradientBoosting (untuned) is the best model** with F1 = 0.7593. Tuning actually degraded performance slightly (0.7552 vs 0.7593), despite a higher CV F1 of 0.7643 — this gap indicates mild overfitting to CV folds.
- All models exhibit **high precision (~0.83--0.87) but moderate recall (~0.67--0.69)**, which is typical behavior on imbalanced data where the model learns to be conservative in predicting the minority class.
- Logistic Regression tuning yields virtually no improvement (0.7544 vs 0.7548), confirming the linear model is near its capacity on this data.
- Random Forest has the lowest F1 (0.7432) and the lowest precision (0.8305) among all models.

# 6. Threshold Optimization

**Business context:** A false negative (missing a potential subscriber) costs more than a false positive (sending an unnecessary follow-up). We should optimize the decision threshold to favor recall while maintaining acceptable precision.

The default threshold is 0.5, but we can lower it to catch more true positives at the cost of some false positives.

**Note on metric choice:** The challenge uses F1-score, which balances precision and recall equally. However, from a pure business standpoint, recall matters more here: the cost of missing a potential subscriber (false negative) far exceeds the cost of targeting a non-subscriber (false positive -- e.g., showing an extra popup). We optimize the threshold on F1 to comply with the challenge metric, but acknowledge that a recall-biased threshold (e.g., 0.3-0.4) would be preferable in a production deployment.

In [20]:
# Select the model with best F1 for threshold tuning
best_model_name = results_df.iloc[-1]["Model"]
best_pipeline = fitted_models[best_model_name]
print(f"Best model: {best_model_name}")

# Get predicted probabilities
y_val_proba = best_pipeline.predict_proba(X_val)[:, 1]

# Compute precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(
    y_val, y_val_proba,
)

# Compute F1 for each threshold
f1_scores = (
    2 * (precisions * recalls) / (precisions + recalls + 1e-10)
)

Best model: GradientBoosting


In [21]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Precision-Recall Curve",
        "F1 / Precision / Recall vs Threshold",
    ),
)

fig.add_trace(
    go.Scatter(
        x=recalls, y=precisions, mode="lines",
        line=dict(color="#3498db", width=2), name="PR Curve",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=thresholds, y=precisions[:-1], mode="lines",
        line=dict(color="#3498db"), name="Precision",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=thresholds, y=recalls[:-1], mode="lines",
        line=dict(color="#e74c3c"), name="Recall",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=thresholds, y=f1_scores[:-1], mode="lines",
        line=dict(color="#2ecc71", width=3), name="F1",
    ),
    row=1, col=2,
)

best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]
best_f1_at_threshold = f1_scores[best_idx]

fig.add_vline(
    x=best_threshold, line_dash="dash", line_color="gray",
    row=1, col=2,
)
fig.add_annotation(
    x=best_threshold, y=best_f1_at_threshold,
    text=(
        f"Best F1={best_f1_at_threshold:.3f}"
        f"<br>threshold={best_threshold:.3f}"
    ),
    row=1, col=2,
)

fig.update_layout(
    height=450,
    title_text=f"Threshold Analysis ({best_model_name})",
)
fig.update_xaxes(title_text="Recall", row=1, col=1)
fig.update_yaxes(title_text="Precision", row=1, col=1)
fig.update_xaxes(title_text="Threshold", row=1, col=2)
fig.update_yaxes(title_text="Score", row=1, col=2)
fig.write_image(PLOT_DIR / "11_precision_recall_curve.png", width=1200, height=450, scale=2)
fig.show()

print(f"\nOptimal threshold (max F1): {best_threshold:.4f}")
print(f"F1 at optimal threshold: {best_f1_at_threshold:.4f}")
print(f"Precision at optimal: {precisions[best_idx]:.4f}")
print(f"Recall at optimal: {recalls[best_idx]:.4f}")


Optimal threshold (max F1): 0.4459
F1 at optimal threshold: 0.7629
Precision at optimal: 0.8542
Recall at optimal: 0.6892


### Expected output

![11_precision_recall_curve](reports/figures/11_precision_recall_curve.png)

- The optimal threshold is **0.4459** — below the default 0.5. This is expected: given the class imbalance (96.8% vs 3.2%), the model's predicted probabilities concentrate well below 0.5 for most samples, so the best F1 operating point shifts downward.
- At this threshold: **F1 = 0.7629**, Precision = 0.8542, Recall = 0.6892.
- The improvement over the default threshold is modest but consistent: F1 goes from **0.7593 to 0.7629 (+0.0036)**. The gain comes from a small increase in recall (0.6804 to 0.6892) with only a slight precision drop (0.8589 to 0.8542).
- The PR curve shows a reliable operating region where precision remains above 0.85 at moderate recall levels, confirming the model's conservative but accurate behavior.

### Threshold strategy: challenge vs. business

The F1-optimal threshold maximizes the challenge metric. However, in a production context, false negatives (missing a potential subscriber) are arguably more costly than false positives (sending an extra email). The break-even point (where precision equals recall) offers an alternative operating point that balances both types of errors equally.

In [22]:
# Break-even analysis: find threshold where precision equals recall
# WHY: in production, the cost asymmetry (FN > FP) may favor a different
# operating point than the F1-optimal threshold used for the challenge.
diff = np.abs(precisions[:-1] - recalls[:-1])
be_idx = np.argmin(diff)
th_breakeven = thresholds[be_idx]

y_val_pred_be = (y_val_proba >= th_breakeven).astype(int)
f1_be = f1_score(y_val, y_val_pred_be)
p_be = precision_score(y_val, y_val_pred_be)
r_be = recall_score(y_val, y_val_pred_be)

print("Threshold comparison:")
print(f"  {'Strategy':<20} {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'F1-optimal':<20} {best_threshold:>10.4f} {precisions[best_idx]:>10.4f} {recalls[best_idx]:>10.4f} {best_f1_at_threshold:>10.4f}")
print(f"  {'Break-even (P=R)':<20} {th_breakeven:>10.4f} {p_be:>10.4f} {r_be:>10.4f} {f1_be:>10.4f}")
print(f"\nDelta F1: {f1_be - best_f1_at_threshold:+.4f} ({(f1_be - best_f1_at_threshold) / best_f1_at_threshold * 100:+.2f}%)")

Threshold comparison:
  Strategy              Threshold  Precision     Recall         F1
  F1-optimal               0.4459     0.8542     0.6892     0.7629
  Break-even (P=R)         0.3025     0.7512     0.7520     0.7516

Delta F1: -0.0113 (-1.48%)


### Expected output

The break-even threshold provides an alternative operating point where precision equals recall. For the challenge, we use the F1-optimal threshold. In a production setting, the break-even point may be preferred when the cost of missing a subscriber (false negative) outweighs the cost of an extra email (false positive).

In [ ]:
y_val_pred_default = (y_val_proba >= 0.5).astype(int)
f1_default = f1_score(y_val, y_val_pred_default)

y_val_pred_optimized = (y_val_proba >= best_threshold).astype(int)
f1_optimized = f1_score(y_val, y_val_pred_optimized)

print(f"Default threshold (0.5):     F1={f1_default:.4f}")
print(
    f"Optimized threshold ({best_threshold:.3f}): "
    f"F1={f1_optimized:.4f}"
)
print(f"Improvement: +{(f1_optimized - f1_default):.4f}")

labels = ["Not converted", "Converted"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Default (threshold=0.5, F1={f1_default:.3f})",
        f"Optimized (threshold={best_threshold:.3f}, F1={f1_optimized:.3f})",
    ],
)

for col_idx, (preds, title) in enumerate([
    (y_val_pred_default, "Default"),
    (y_val_pred_optimized, "Optimized"),
], start=1):
    cm = confusion_matrix(y_val, preds)
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=labels,
            y=labels,
            text=cm,
            texttemplate="%{text}",
            colorscale="Greens",
            showscale=col_idx == 2,
        ),
        row=1, col=col_idx,
    )
    fig.update_xaxes(title_text="Predicted Label", row=1, col=col_idx)
    fig.update_yaxes(title_text="True Label", row=1, col=col_idx)

fig.update_layout(
    title_text="Confusion Matrix Comparison",
    width=1000,
    height=500,
)
fig.write_image(PLOT_DIR / "12_confusion_matrix.png", width=1000, height=500, scale=2)
fig.show()

print("\nClassification Report (optimized threshold):")
print(classification_report(y_val, y_val_pred_optimized))

### Expected output

![12_confusion_matrix](reports/figures/12_confusion_matrix.png)

- At the optimized threshold (0.446), the model achieves **F1 = 0.7629** (vs 0.7593 at default 0.5).
- **High precision (0.85)** means few false positives — when the model predicts a conversion, it is usually correct. This is valuable in production to avoid wasting targeting resources on unlikely converters.
- **Moderate recall (0.69)** means we miss approximately 31% of actual converters. This is the main limitation and a direct consequence of the class imbalance (96.8% non-converters vs 3.2% converters).
- Classification report at optimized threshold: Class 0 (P=0.99, R=1.00, F1=0.99, n=30600) vs Class 1 (P=0.85, R=0.69, F1=0.76, n=1020). The asymmetry reflects the imbalance — the model is very strong on the majority class but trades off recall on the minority class to maintain high precision.

# 7. Feature Importance

In [24]:
feature_names = []
cat_encoder = (
    best_pipeline
    .named_steps["preprocessor"]
    .named_transformers_["cat"]
)
feature_names.extend(
    cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES)
)
feature_names.extend(NUMERICAL_FEATURES)
feature_names.extend(BINARY_FEATURES)

importances = (
    best_pipeline.named_steps["classifier"].feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances,
}).sort_values("Importance", ascending=True)

fig = go.Figure(
    go.Bar(
        x=importance_df["Importance"],
        y=importance_df["Feature"],
        orientation="h",
        marker_color="#3498db",
    )
)
fig.update_layout(
    title=f"Feature Importances ({best_model_name})",
    height=500,
    xaxis_title="Importance",
    yaxis_title="Feature",
)
fig.write_image(PLOT_DIR / "13_feature_importance.png", width=800, height=500, scale=2)
fig.show()

print("Top 5 features:")
print(importance_df.tail(5).to_string(index=False))

Top 5 features:
            Feature  Importance
                age    0.013231
   new_user_x_pages    0.022663
           new_user    0.023005
           is_china    0.039576
total_pages_visited    0.886958


### Expected output

![13_feature_importance](reports/figures/13_feature_importance.png)

- **`total_pages_visited` dominates overwhelmingly (0.887, ~89% of total importance)**, confirming that page engagement depth is the single most actionable business lever for conversion.
- `is_china` is a distant second (0.040), confirming the China anomaly detected in EDA as a meaningful signal — geographic origin matters, but far less than engagement depth.
- `new_user` (0.023) and `new_user_x_pages` (0.023) contribute nearly equally. The interaction feature does not add much beyond the base variable, suggesting the new-user effect is largely additive rather than multiplicative with page visits.
- `age` has minimal importance (0.013), consistent with EDA findings that age has weak predictive power for conversion.
- The model is essentially a **"pages visited" classifier with minor corrections** from geography and user type. Traffic source features have near-zero importance, indicating conversion is driven by on-site behavior rather than acquisition channel.

# 8. Final Model Training & Predictions

In [25]:
from sklearn.base import clone

# Clone the best pipeline (preprocessor + classifier together)
best_pipeline_final = clone(best_pipeline)

# Refit on ALL training data (X_full, y_full)
best_pipeline_final.fit(X_full, y_full)

# Generate test predictions with optimized threshold
test_proba = best_pipeline_final.predict_proba(
    X_test_final,
)[:, 1]
test_predictions = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({TARGET: test_predictions})
output_path = (
    str(TRAIN_FILE)
    .replace("train", "test_predictions_final")
    .replace("input", "output")
)
submission.to_csv(output_path, index=False)

print(f"Predictions: {len(submission)} rows")
print(
    f"Predicted conversion rate: "
    f"{submission[TARGET].mean():.4f}"
)
print(f"Distribution:\n{submission[TARGET].value_counts()}")

Predictions: 31620 rows
Predicted conversion rate: 0.0265
Distribution:
converted
0    30781
1      839
Name: count, dtype: int64


In [26]:
import os

os.makedirs(MODELS_DIR, exist_ok=True)

# Save the full pipeline (preprocessor + classifier)
joblib.dump(best_pipeline_final, MODEL_FILE)

# Also save preprocessor separately (for inspection)
joblib.dump(
    best_pipeline_final.named_steps["preprocessor"],
    PREPROCESSOR_FILE,
)

joblib.dump(
    {
        "model_name": best_model_name,
        "best_threshold": float(best_threshold),
        "val_f1_default": float(f1_default),
        "val_f1_optimized": float(f1_optimized),
        "val_precision": float(precisions[best_idx]),
        "val_recall": float(recalls[best_idx]),
        "feature_importance": importance_df.to_dict(
            orient="records",
        ),
        "results_comparison": results_df.to_dict(
            orient="records",
        ),
        "best_params": (
            best_params
            if "tuned" in best_model_name
            else {}
        ),
    },
    METADATA_FILE,
)

print(f"Model saved to {MODEL_FILE}")
print(f"Preprocessor saved to {PREPROCESSOR_FILE}")
print(f"Metadata saved to {METADATA_FILE}")

Model saved to models/best_model.joblib
Preprocessor saved to models/preprocessor.joblib
Metadata saved to models/metadata.joblib


# 9. Business Recommendations

## Model Summary

| Aspect | Value |
|--------|-------|
| Best Model | GradientBoosting (untuned baseline) |
| Validation F1 (default threshold) | 0.7593 |
| Validation F1 (optimized threshold) | 0.7629 |
| Optimal Threshold | 0.4459 |
| Precision (at optimal) | 0.8542 |
| Recall (at optimal) | 0.6892 |
| Top Features | total_pages_visited (0.887), is_china (0.040), new_user (0.023) |

The untuned GradientBoosting baseline outperforms the tuned version (F1 0.7593 vs 0.7552), indicating the tuning overfit to CV folds.

## Why Threshold Tuning Matters

The default 0.5 threshold treats false positives and false negatives equally. In the newsletter context:
- **False negative** = a visitor who would have subscribed but we failed to identify. **Lost opportunity.**
- **False positive** = a visitor we targeted who won't subscribe. **Low cost** (an extra email).

The optimized threshold (0.4459) shifts downward from 0.5 — expected given the class imbalance where predicted probabilities cluster below 0.5. The result: F1 = 0.7629, Precision = 0.8542, Recall = 0.6892.

## Feature Analysis & Actionable Recommendations

### 1. Total Pages Visited (Dominant Feature -- ~89% of importance)
Users who browse more pages convert at dramatically higher rates. This is overwhelmingly the #1 lever.
- **Action:** Improve internal linking, add "related articles" sections, optimize content to reduce bounce rate
- **Action:** Implement progressive engagement -- show newsletter CTA after N pages viewed
- **KPI to track:** Average pages per session

### 2. New User vs Returning User
Returning visitors convert ~5x more than first-time visitors (7.2% vs 1.4%). Feature importance is low (0.023) but the business signal is clear.
- **Action:** Implement retargeting campaigns for first-time visitors
- **Action:** Add cookie-based personalization to recognize returning visitors and show targeted CTAs
- **Action:** Email re-engagement sequences for visitors who browsed but didn't subscribe

### 3. China Investigation (2nd most important feature, 0.040)
China has 0.1% conversion vs 6.2% for Germany -- a 60x gap. `is_china` is the 2nd most important feature in the model.
- **Action:** Investigate whether this is a localization issue, product-market fit problem, or data quality issue (bots, unqualified traffic)
- **Action:** If legitimate traffic: localize content and CTAs for the Chinese market
- **Action:** If data quality issue: filter or flag suspicious traffic before modeling

### 4. Traffic Source (negligible predictive power)
Traffic source has near-zero feature importance in the model. Conversion differences across sources are small (Ads 3.5%, SEO 3.3%, Direct 2.8%).
- **Observation:** Source-level optimizations are unlikely to improve conversion rates based on current data.
- **Note:** SEO has only 3.3% conversion rate, which is not materially different from other channels — not actionable as a lever.
- **Action:** Focus budget on driving volume rather than optimizing by source.

### 5. Age Patterns (marginal importance)
Age has very low importance (0.013). This is a marginal lever at best.
- **Action:** Low-priority A/B test for age-appropriate messaging, but do not expect significant impact.

## Priority Matrix

| Action | Impact | Effort | Priority |
|--------|--------|--------|----------|
| Increase pages per session | Very High (~89% importance) | Medium | **P0** |
| Progressive newsletter CTA | Very High | Low | **P0** |
| Investigate China anomaly (2nd feature, 0.1% conv) | Medium | Low | **P1** |
| Returning visitor retention (7.2% vs 1.4%) | Medium | Medium | **P1** |
| Threshold optimization in production | Low | Low | **P2** |
| Age-based personalization | Marginal | Medium | **P3** |

### Threshold Strategy: Competition vs. Business

**For the competition**, we use the F1-optimal threshold (0.4459) that maximizes our validation F1 at 0.7629.

**For production deployment**, the choice depends on cost asymmetry:
- At **threshold = 0.4459**: Precision = 0.8542, Recall = 0.6892. High precision means fewer wasted emails, but ~31% of actual converters are missed.
- **If false negatives are costly** (lost subscribers), a lower threshold increases recall at the expense of precision. The break-even point (precision = recall) is at threshold 0.3025 (F1 = 0.7516, both metrics at ~0.75).
- **If false positives are costly** (email fatigue, unsubscribes), the current high-precision threshold is appropriate.
- Test predictions at the optimized threshold yield 2.65% positive rate (839 out of 31620), consistent with the ~3.2% base rate.

This dual perspective demonstrates that the optimal threshold depends on the business context, not just the mathematical optimum.